In [16]:
!pip install transformers datasets accelerate trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 15.9 MB/s eta 0:00:00


In [5]:
import pandas as pd

# Login using e.g. `huggingface-cli login` to access this dataset
df = pd.read_csv("hf://datasets/SnehaDeshmukh/IndianBailJudgments-1200/indian_bail_judgments.csv")

In [6]:
print(df.head())

   case_id                                         case_title  \
0        1    Jibangshu Paul vs National Investigation Agency   
1        2  State By The Superintendent Of Police vs Mehbo...   
2        3                        Hyderali vs State Of Kerala   
3        4  Rashbehari Karmakar vs Indrajit Mukherjee And ...   
4        5  Smt. Shankri Devi Age 62 Years vs Union Territ...   

                                               court        date  \
0                                 Gauhati High Court  2011-07-27   
1                                  Madras High Court  1997-09-30   
2                                  Kerala High Court  2008-08-05   
3                                Calcutta High Court  2002-07-12   
4  High Court of Jammu & Kashmir and Ladakh at Jammu  2023-06-12   

                                         judge  \
0  Justice I. A. Ansari, Justice A. K. Goswami   
1                       Justice E. Padmanabhan   
2                              Justice K. Hema   


In [7]:
print(df.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 25 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   case_id                      1200 non-null   int64 
 1   case_title                   1200 non-null   object
 2   court                        1200 non-null   object
 3   date                         1200 non-null   object
 4   judge                        1200 non-null   object
 5   ipc_sections                 1200 non-null   object
 6   bail_type                    1200 non-null   object
 7   bail_cancellation_case       1200 non-null   bool  
 8   landmark_case                1200 non-null   bool  
 9   accused_name                 1200 non-null   object
 10  accused_gender               1200 non-null   object
 11  prior_cases                  1200 non-null   object
 12  bail_outcome                 1200 non-null   object
 13  bail_outcome_label_detailed  1200

In [3]:
df.to_csv("bail_dataset.csv", index=False)

In [9]:
import json
import pandas as pd

with open("/content/IndicLegalQA Dataset_10K.json", "r") as f:
    data = json.load(f)

FileNotFoundError: [Errno 2] No such file or directory: '/content/IndicLegalQA Dataset_10K.json'

In [10]:
print(type(data))
print(data[0])   # check first sample

NameError: name 'data' is not defined

In [11]:
df_qa = pd.DataFrame(data)

NameError: name 'data' is not defined

In [22]:
grouped = df_qa.groupby("case_name")

In [13]:
aggregated_data = []

for case, group in grouped:
    qa_list = []

    for _, row in group.iterrows():
        qa_list.append({
            "question": row["question"],
            "answer": row["answer"]
        })

    aggregated_data.append({
        "case_name": case,
        "judgment_date": group["judgment_date"].iloc[0],
        "qa_pairs": qa_list
    })

In [14]:
df_grouped = pd.DataFrame(aggregated_data)

In [15]:
print(df_grouped.head(1))

                                      case_name    judgment_date  \
0  A. Sreenivasa Reddy vs. Rakesh Sharma & Anr.  8th August 2023   

                                            qa_pairs  
0  [{'question': 'What were the charges against A...  


In [16]:
def convert_qa_pairs_to_text(qa_list):
    qa_text = ""

    for i, qa in enumerate(qa_list):
        question = qa.get("question", "")
        answer = qa.get("answer", "")

        qa_text += f"Q{i+1}: {question}\n"
        qa_text += f"A{i+1}: {answer}\n\n"

    return qa_text.strip()

In [17]:
df_grouped["qa_text"] = df_grouped["qa_pairs"].apply(convert_qa_pairs_to_text)

In [18]:
 df_grouped = df_grouped.drop(columns=["qa_pairs"])

In [19]:
pd.set_option('display.max_colwidth', None)
print(df_grouped.head(1))

                                      case_name    judgment_date  \
0  A. Sreenivasa Reddy vs. Rakesh Sharma & Anr.  8th August 2023   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

In [20]:
def convert_to_structured_qa(qa_list):
    text = ""

    for i, qa in enumerate(qa_list):
        text += f"Step {i+1} - Question: {qa['question']}\n"
        text += f"Step {i+1} - Answer: {qa['answer']}\n\n"

    return text.strip()

In [23]:
df_grouped = pd.DataFrame(aggregated_data)
df_grouped["qa_text"] = df_grouped["qa_pairs"].apply(convert_to_structured_qa)

In [24]:
df_grouped["model_input"] = (
    "Case Name: " + df_grouped["case_name"] + "\n" +
    "Judgment Date: " + df_grouped["judgment_date"] + "\n\n" +
    "Legal Reasoning:\n" + df_grouped["qa_text"]
)

In [4]:
pd.set_option('display.max_colwidth', None)
print(df_grouped[["model_input"]].head(1))

NameError: name 'pd' is not defined

In [3]:
pd.set_option('display.max_colwidth', None)
print(df_grouped.head())

NameError: name 'pd' is not defined

In [27]:
 df_grouped.drop(columns=["qa_pairs"], inplace=True)
df_grouped.to_csv("final_structured_qa_dataset.csv", index=False)

In [28]:
df_grouped.drop(columns=["model_input"], inplace=True)

In [29]:
df_grouped.to_csv("final_structured_qa_dataset.csv", index=False)

In [13]:
import pandas as pd

bail_df = pd.read_csv("bail_dataset.csv")
print(bail_df.head())

   case_id                                         case_title  \
0        1    Jibangshu Paul vs National Investigation Agency   
1        2  State By The Superintendent Of Police vs Mehbo...   
2        3                        Hyderali vs State Of Kerala   
3        4  Rashbehari Karmakar vs Indrajit Mukherjee And ...   
4        5  Smt. Shankri Devi Age 62 Years vs Union Territ...   

                                               court        date  \
0                                 Gauhati High Court  2011-07-27   
1                                  Madras High Court  1997-09-30   
2                                  Kerala High Court  2008-08-05   
3                                Calcutta High Court  2002-07-12   
4  High Court of Jammu & Kashmir and Ladakh at Jammu  2023-06-12   

                                         judge  \
0  Justice I. A. Ansari, Justice A. K. Goswami   
1                       Justice E. Padmanabhan   
2                              Justice K. Hema   


In [12]:
qa_df = pd.read_csv("final_structured_qa_dataset.csv")
print(qa_df.head())

                                           case_name    judgment_date  \
0       A. Sreenivasa Reddy vs. Rakesh Sharma & Anr.  8th August 2023   
1  A.G. Perarivalan vs State, Through Superintend...              NaN   
2  AMEET LALCHAND SHAH AND OTHERS v. RISHABH ENTE...              NaN   
3  APS Forex Services Pvt. Ltd. vs Shakti Interna...              NaN   
4     Aarif & Ors. vs. The State of Rajasthan & Anr.              NaN   

                                             qa_text  
0  Step 1 - Question: What were the charges again...  
1  Step 1 - Question: What was the appellant A.G....  
2  Step 1 - Question: What is the main subject of...  
3  Step 1 - Question: What was the primary legal ...  
4  Step 1 - Question: What were the charges again...  


In [14]:
bail_df["text"] = (
    "Facts: " + bail_df["facts"].fillna('') + "\n\n" +
    "Judgment Reason: " + bail_df["judgment_reason"].fillna('')
)

In [15]:
def extract_label(summary):
    summary = str(summary).lower()

    if "granted" in summary:
        return 1
    elif "rejected" in summary or "dismissed" in summary:
        return 0
    else:
        return 0  # default

bail_df["label"] = bail_df["summary"].apply(extract_label)

In [16]:
print(bail_df["label"].value_counts())

label
1    683
0    517
Name: count, dtype: int64


In [17]:
qa_df["text"] = qa_df["qa_text"]

In [18]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("law-ai/InCaseLawBERT")

def tokenize(texts):
    return tokenizer(
        texts.tolist(),
        padding=True,
        truncation=True,
        max_length=512
    )

In [30]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "law-ai/InCaseLawBERT"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: law-ai/InCaseLawBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect 

In [19]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    bail_df[["text", "label"]],
    test_size=0.2,
    random_state=42
)

In [20]:
def tokenize_data(df):
    return tokenizer(
        df["text"].tolist(),
        padding=True,
        truncation=True,
        max_length=512
    )

train_encodings = tokenize_data(train_df)
test_encodings = tokenize_data(test_df)

In [21]:
import torch

class BailDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.reset_index(drop=True)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = BailDataset(train_encodings, train_df["label"])
test_dataset = BailDataset(test_encodings, test_df["label"])

In [22]:
!pip install -U transformers

In [25]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",   # ✅ NEW NAME (not evaluation_strategy)
    save_strategy="epoch",
    logging_steps=50,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    load_best_model_at_end=True
)

In [28]:
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)

    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds)

    return {
        "accuracy": acc,
        "f1": f1
    }

In [32]:
from transformers import Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [33]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.435375,0.333920,0.916667,0.923664
2,0.251972,0.415407,0.904167,0.918728
3,0.109997,0.409073,0.904167,0.916364


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=720, training_loss=0.31557832260926566, metrics={'train_runtime': 186.6828, 'train_samples_per_second': 15.427, 'train_steps_per_second': 3.857, 'total_flos': 319679932262400.0, 'train_loss': 0.31557832260926566, 'epoch': 3.0})

In [42]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    load_best_model_at_end=True,
    disable_tqdm=True   # 🔥 IMPORTANT FIX
)

In [44]:
from transformers import Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [45]:
trainer.train()

{'loss': '0.2363', 'grad_norm': '17.88', 'learning_rate': '4.66e-05', 'epoch': '0.2083'}
{'loss': '0.1503', 'grad_norm': '0.07295', 'learning_rate': '4.313e-05', 'epoch': '0.4167'}
{'loss': '0.1069', 'grad_norm': '0.00713', 'learning_rate': '3.965e-05', 'epoch': '0.625'}
{'loss': '0.1409', 'grad_norm': '106.9', 'learning_rate': '3.618e-05', 'epoch': '0.8333'}
{'eval_loss': '0.5899', 'eval_accuracy': '0.8875', 'eval_f1': '0.9032', 'eval_runtime': '3.297', 'eval_samples_per_second': '72.8', 'eval_steps_per_second': '18.2', 'epoch': '1'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '0.0678', 'grad_norm': '0.0113', 'learning_rate': '3.271e-05', 'epoch': '1.042'}
{'loss': '0.1431', 'grad_norm': '0.007683', 'learning_rate': '2.924e-05', 'epoch': '1.25'}
{'loss': '0.09972', 'grad_norm': '35.1', 'learning_rate': '2.576e-05', 'epoch': '1.458'}
{'loss': '0.04421', 'grad_norm': '0.004289', 'learning_rate': '2.229e-05', 'epoch': '1.667'}
{'loss': '0.08328', 'grad_norm': '0.02263', 'learning_rate': '1.882e-05', 'epoch': '1.875'}
{'eval_loss': '0.6702', 'eval_accuracy': '0.8833', 'eval_f1': '0.9014', 'eval_runtime': '3.266', 'eval_samples_per_second': '73.48', 'eval_steps_per_second': '18.37', 'epoch': '2'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '0.09315', 'grad_norm': '0.01001', 'learning_rate': '1.535e-05', 'epoch': '2.083'}
{'loss': '0.04308', 'grad_norm': '0.00229', 'learning_rate': '1.188e-05', 'epoch': '2.292'}
{'loss': '0.03043', 'grad_norm': '0.006999', 'learning_rate': '8.403e-06', 'epoch': '2.5'}
{'loss': '0.0001361', 'grad_norm': '0.003167', 'learning_rate': '4.931e-06', 'epoch': '2.708'}
{'loss': '0.01074', 'grad_norm': '0.001523', 'learning_rate': '1.458e-06', 'epoch': '2.917'}
{'eval_loss': '0.8082', 'eval_accuracy': '0.8833', 'eval_f1': '0.9014', 'eval_runtime': '3.334', 'eval_samples_per_second': '71.98', 'eval_steps_per_second': '17.99', 'epoch': '3'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '291.1', 'train_samples_per_second': '9.894', 'train_steps_per_second': '2.473', 'train_loss': '0.08681', 'epoch': '3'}


There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=720, training_loss=0.0868129693244959, metrics={'train_runtime': 291.0879, 'train_samples_per_second': 9.894, 'train_steps_per_second': 2.473, 'train_loss': 0.0868129693244959, 'epoch': 3.0})

In [46]:
results = trainer.evaluate()
print(results)

{'eval_loss': '0.5901', 'eval_accuracy': '0.8875', 'eval_f1': '0.9032', 'eval_runtime': '3.278', 'eval_samples_per_second': '73.21', 'eval_steps_per_second': '18.3', 'epoch': '3'}
{'eval_loss': 0.5900592803955078, 'eval_accuracy': 0.8875, 'eval_f1': 0.9032258064516129, 'eval_runtime': 3.2781, 'eval_samples_per_second': 73.213, 'eval_steps_per_second': 18.303, 'epoch': 3.0}


In [54]:
trainer.save_model("final_bail_model")
tokenizer.save_pretrained("final_bail_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('final_bail_model/tokenizer_config.json', 'final_bail_model/tokenizer.json')

In [5]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_path = "final_bail_model"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [6]:
def split_text(text, max_len=400):
    words = text.split()
    return [" ".join(words[i:i+max_len]) for i in range(0, len(words), max_len)]

In [7]:
def predict_bail(text):
    chunks = split_text(text)
    preds = []

    for chunk in chunks:
        inputs = tokenizer(chunk, return_tensors="pt", truncation=True, padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)

        probs = torch.softmax(outputs.logits, dim=1)
        preds.append(probs.cpu().numpy())

    final_pred = sum(preds) / len(preds)

    return "Granted" if final_pred.argmax() == 1 else "Rejected"

In [8]:
def get_important_sentences(text):
    sentences = text.split(".")
    base_pred = predict_bail(text)

    important = []

    for i, sent in enumerate(sentences):
        temp = sentences[:i] + sentences[i+1:]
        new_text = ".".join(temp)

        new_pred = predict_bail(new_text)

        if new_pred != base_pred:
            important.append(sent.strip())

    return important[:3]

In [9]:
import pandas as pd

qa_df = pd.read_csv("final_structured_qa_dataset.csv")

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer(max_features=5000)
qa_vectors = vectorizer.fit_transform(qa_df["qa_text"])

def get_similar_reasoning(text):
    input_vec = vectorizer.transform([text])
    similarities = cosine_similarity(input_vec, qa_vectors)
    idx = similarities.argmax()
    return qa_df.iloc[idx]["qa_text"]

In [11]:
def final_legal_system(text):
    prediction = predict_bail(text)
    reasoning = get_similar_reasoning(text)
    important = get_important_sentences(text)

    return {
        "prediction": prediction,
        "reasoning": reasoning,
        "important_sentences": important
    }

In [12]:
import pandas as pd

bail_df = pd.read_csv("bail_dataset.csv")
bail_df["text"] = (
    "Facts: " + bail_df["facts"].fillna('') + "\n\n" +
    "Judgment Reason: " + bail_df["judgment_reason"].fillna('')
)

def extract_label(summary):
    summary = str(summary).lower()

    if "granted" in summary:
        return 1
    elif "rejected" in summary or "dismissed" in summary:
        return 0
    else:
        return 0  # default

bail_df["label"] = bail_df["summary"].apply(extract_label)

sample = bail_df["text"].iloc[5]

result = final_legal_system(sample)

print("Prediction:", result["prediction"])
print("\nReasoning:\n", result["reasoning"])
print("\nImportant Sentences:\n", result["important_sentences"])


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Prediction: Granted

Reasoning:
 Step 1 - Question: What was the primary concern addressed in the case of Sampurna Behura vs Union of India & Ors.?
Step 1 - Answer: The primary concern was the virtual non-implementation or tardy implementation of laws beneficial to children, particularly the Juvenile Justice (Care and Protection of Children) Act, 2000 and the Juvenile Justice (Care and Protection of Children) Act, 2015.

Step 2 - Question: What efforts were appreciated by the Supreme Court in this case?
Step 2 - Answer: The Supreme Court appreciated the efforts of Sampurna Behura for highlighting the issues related to the implementation of juvenile justice laws through a Public Interest Litigation and acknowledged the assistance rendered by the learned counsel for the appearing parties.

Step 3 - Question: What did the Chief Justices' Conferences resolve regarding the juvenile justice system?
Step 3 - Answer: The Chief Justices' Conferences resolved to expedite the setting up of Juveni

In [13]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

cot_model_name = "google/flan-t5-base"

cot_tokenizer = AutoTokenizer.from_pretrained(cot_model_name)
cot_model = AutoModelForSeq2SeqLM.from_pretrained(cot_model_name)

cot_model.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=768, out_features=2048, bias=False)
              (wi_1): Linear(in_features=768, out_features=2048, bias=False)
              (wo):

In [35]:
def generate_cot(text):
    prompt = f"""
Analyze this legal case and provide step-by-step reasoning.

Case:
{text}

Step-by-step reasoning:
"""

    inputs = cot_tokenizer(prompt, return_tensors="pt", truncation=True).to(device)

    outputs = cot_model.generate(**inputs, max_length=300)

    return cot_tokenizer.decode(outputs[0], skip_special_tokens=True)

In [15]:
def reward_function(reasoning):
    score = 0

    if "step" in reasoning.lower():
        score += 1

    if "final" in reasoning.lower():
        score += 1

    if len(reasoning) > 100:
        score += 0.5

    return score

In [16]:
def improve_reasoning(text):
    best_reasoning = ""
    best_score = -1

    # generate multiple reasoning outputs
    for _ in range(3):
        reasoning = generate_cot(text)
        score = reward_function(reasoning)

        if score > best_score:
            best_score = score
            best_reasoning = reasoning

    return best_reasoning

In [17]:
def final_legal_system(text):
    prediction = predict_bail(text)
    important = get_important_sentences(text)

    # RL-improved reasoning
    reasoning = improve_reasoning(text)

    return {
        "prediction": prediction,
        "reasoning": reasoning,
        "important_sentences": important
    }

In [36]:
sample = bail_df["text"].iloc[5]

result = final_legal_system(sample)

print("Prediction:", result["prediction"])
print("\nReasoning:\n", result["reasoning"])
print("\nImportant:\n", result["important_sentences"])

Prediction: Granted

Reasoning:
 Step 1: The court emphasized that Section 12 of the JJ Act mandates bail for juveniles unless release would expose them to danger or defeat justice Step 2: The petitioner had not inflicted fatal injuries and co-accused were already in custody, so no such danger remained Step 3: The court emphasized that Section 12 of the JJ Act mandates bail for juveniles unless release would expose them to danger or defeat justice
Final Answer: Based on above reasoning.

Important:
 ['Facts: Hardeep Singh @ Dipi, a juvenile, was implicated in a gang-related robbery and murder based on co-accused confessions', ' He had been in a juvenile protection home for almost a year', ' Previous courts denied bail citing societal risk and psychological harm']


In [19]:
 def clean_reasoning(text):
    sentences = text.split(".")

    seen = set()
    unique_sentences = []

    for s in sentences:
        s = s.strip()
        if s and s not in seen:
            seen.add(s)
            unique_sentences.append(s)

    return ". ".join(unique_sentences[:5])  # limit length

In [34]:
def generate_cot(text):
    important = get_important_sentences(text)
    facts = " ".join(important)

    prompt = f"""
You are a legal reasoning system.

ONLY use the facts given below.
DO NOT write generic explanations.
DO NOT say "purpose" or "this article".

Facts:
{facts}

Write reasoning strictly like this:

Step 1: Mention facts from case
Step 2: Apply legal rule (JJ Act or bail principles)
Step 3: Explain why bail is granted/rejected
Final Answer:

"""

    inputs = cot_tokenizer(prompt, return_tensors="pt", truncation=True).to(device)

    outputs = cot_model.generate(
        **inputs,
        max_length=200,
        repetition_penalty=2.0,        # 🔥 stronger
        no_repeat_ngram_size=4,        # 🔥 stricter
        temperature=0.7                # 🔥 less randomness
    )

    result = cot_tokenizer.decode(outputs[0], skip_special_tokens=True)

    return result

In [21]:
def get_important_sentences(text):
    sentences = text.split(".")

    base_pred = predict_bail(text)
    important = []

    for i, sent in enumerate(sentences):
        if len(sent.strip()) < 20:
            continue  # skip small sentences

        temp = sentences[:i] + sentences[i+1:]
        new_text = ".".join(temp)

        new_pred = predict_bail(new_text)

        if new_pred != base_pred:
            important.append(sent.strip())

    # fallback (IMPORTANT)
    if len(important) == 0:
        important = sentences[:3]

    return important[:3]

In [38]:
def format_reasoning(reasoning):
    sentences = [s.strip() for s in reasoning.split(".") if len(s.strip()) > 20]

    unique = []
    seen = set()

    for s in sentences:
        if s not in seen:
            seen.add(s)
            unique.append(s)

    steps = []

    if len(unique) >= 3:
        steps.append(f"Step 1: {unique[0]}")
        steps.append(f"Step 2: {unique[1]}")
        steps.append(f"Step 3: {unique[2]}")
    else:
        for i, s in enumerate(unique):
            steps.append(f"Step {i+1}: {s}")

    return " ".join(steps) + "\nFinal Answer: Bail Granted based on above reasoning."

In [39]:
def enforce_structure(reasoning, important):
    facts = important[0] if len(important) > 0 else ""
    law = "Section 12 of the JJ Act mandates bail unless there is risk to justice or society."

    return f"""
Step 1: {facts}
Step 2: {law}
Step 3: Based on the case facts and lack of risk, bail is justified.
Final Answer: Bail Granted.
"""

In [28]:
def final_legal_system(text):
    prediction = predict_bail(text)
    important = get_important_sentences(text)

    raw_reasoning = generate_cot(text)
    reasoning = format_reasoning(raw_reasoning)

    return {
        "prediction": prediction,
        "reasoning": reasoning,
        "important_sentences": important
    }

In [40]:
custom_text = """
The accused is a 19-year-old student involved in a minor theft case.
He has no prior criminal record and has been in custody for 2 months.
The prosecution argued risk of repetition, but no strong evidence was found.
"""

result = final_legal_system(custom_text)

print("Prediction:", result["prediction"])
print("\nReasoning:\n", result["reasoning"])
print("\nImportant:\n", result["important_sentences"])

Prediction: Granted

Reasoning:
 Step 1: The accused has no prior criminal record and has been in custody for 2 months Step 2: The accused has no prior criminal record and has no prior criminal record
Final Answer: Bail Granted based on above reasoning.

Important:
 ['\nThe accused is a 19-year-old student involved in a minor theft case', ' \nHe has no prior criminal record and has been in custody for 2 months', ' \nThe prosecution argued risk of repetition, but no strong evidence was found']


next method


In [8]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModel, AutoModelForSeq2SeqLM
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 🔹 Load classification model (InCaseLawBERT)
clf_tokenizer = AutoTokenizer.from_pretrained("final_bail_model")
clf_model = AutoModelForSequenceClassification.from_pretrained("final_bail_model")
clf_model.to(device)
clf_model.eval()

# 🔹 Load embedding model (InCaseLawBERT)
embed_tokenizer = AutoTokenizer.from_pretrained("law-ai/InCaseLawBERT")
embed_model = AutoModel.from_pretrained("law-ai/InCaseLawBERT")
embed_model.to(device)
embed_model.eval()

# 🔹 Load CoT model (FLAN-T5)
cot_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
cot_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")
cot_model.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=768, out_features=2048, bias=False)
              (wi_1): Linear(in_features=768, out_features=2048, bias=False)
              (wo):

In [9]:
import pandas as pd

qa_df = pd.read_csv("final_structured_qa_dataset.csv")

qa_texts = qa_df["qa_text"].tolist()

In [10]:
import re

def sentence_chunking(text, max_sent=3):
    sentences = re.split(r'(?<=[.!?]) +', text)

    chunks = []
    for i in range(0, len(sentences), max_sent):
        chunk = " ".join(sentences[i:i+max_sent])
        chunks.append(chunk)

    return chunks

In [11]:
def get_embedding(text):
    inputs = embed_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512   # 🔥 MUST ADD
    ).to(device)

    with torch.no_grad():
        outputs = embed_model(**inputs)

    emb = outputs.last_hidden_state.mean(dim=1)

    return emb.cpu()

In [12]:
import numpy as np

qa_embeddings = []

for text in qa_texts:
    emb = get_embedding(text)
    qa_embeddings.append(emb.numpy())

qa_embeddings = np.vstack(qa_embeddings)

In [13]:
from sklearn.metrics.pairwise import cosine_similarity

def retrieve_chunks(query, top_k=3):
    query_emb = get_embedding(query).numpy()

    sims = cosine_similarity(query_emb, qa_embeddings)

    top_idx = sims.argsort()[0][-top_k:][::-1]

    return [qa_texts[i] for i in top_idx]

In [14]:
def predict_bail(text):
    inputs = clf_tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = clf_model(**inputs)

    probs = torch.softmax(outputs.logits, dim=1)
    pred = torch.argmax(probs, dim=1).item()

    return "Granted" if pred == 1 else "Rejected"

In [15]:
def get_important_sentences(text):
    sentences = text.split(".")
    base_pred = predict_bail(text)

    important = []

    for i, sent in enumerate(sentences):
        if len(sent.strip()) < 20:
            continue

        temp = sentences[:i] + sentences[i+1:]
        new_text = ".".join(temp)

        new_pred = predict_bail(new_text)

        if new_pred != base_pred:
            important.append(sent.strip())

    if len(important) == 0:
        important = sentences[:3]

    return important[:3]

In [16]:
from datasets import Dataset

def format_example(row):
    return {
        "input": f"""
You are a legal expert.

Case:
{row['case_name']}

Generate step-by-step reasoning:

Step 1:
Step 2:
Step 3:
Final Answer:
""",
        "output": row["qa_text"]
    }

formatted_data = qa_df.apply(format_example, axis=1).tolist()
dataset = Dataset.from_list(formatted_data)

In [17]:
def preprocess(example):
    inputs = cot_tokenizer(
        example["input"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

    labels = cot_tokenizer(
        example["output"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

    inputs["labels"] = labels["input_ids"]
    return inputs

dataset = dataset.map(preprocess, batched=True)

Map:   0%|          | 0/1211 [00:00<?, ? examples/s]

In [1]:
!pip install transformers==4.38.2 peft==0.9.0 trl==0.7.11 accelerate==0.27.2

from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./finetuned_cot",
    per_device_train_batch_size=2,
    num_train_epochs=2,
    logging_steps=50,
    save_strategy="epoch",
    learning_rate=3e-5
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.0/280.0 kB 9.8 MB/s eta 0:00:00
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.13.0
    Uninstalling accelerate-1.13.0:
      Successfully uninstalled accelerate-1.13.0


In [18]:
trainer = Trainer(
    model=cot_model,
    args=training_args,
    train_dataset=dataset
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Step,Training Loss
50,3.265000
100,2.283800
150,2.339100
200,2.212400
250,2.138700
300,1.975900
350,2.011500
400,1.799500
450,1.846800
500,1.826300


TrainOutput(global_step=1212, training_loss=1.9174674477907692, metrics={'train_runtime': 941.4634, 'train_samples_per_second': 2.573, 'train_steps_per_second': 1.287, 'total_flos': 1658482307629056.0, 'train_loss': 1.9174674477907692, 'epoch': 2.0})

In [2]:
def generate_rag_cot(text):
    # 🔹 Retrieve relevant chunks
    retrieved = retrieve_chunks(text)
    context = " ".join(retrieved)

    # 🔹 Add important facts
    important = get_important_sentences(text)
    facts = " ".join(important)

    prompt = f"""
You are a legal reasoning system.

Facts:
{facts}

Context:
{context}

Generate reasoning:

Step 1: Facts
Step 2: Legal principles
Step 3: Analysis
Final Answer:
"""

    inputs = cot_tokenizer(prompt, return_tensors="pt", truncation=True).to(device)

    outputs = cot_model.generate(
        **inputs,
        max_length=300,
        repetition_penalty=1.5,
        no_repeat_ngram_size=3
    )

    return cot_tokenizer.decode(outputs[0], skip_special_tokens=True)

In [3]:
def format_reasoning(reasoning):
    sentences = [s.strip() for s in reasoning.split(".") if len(s.strip()) > 20]

    unique = []
    seen = set()

    for s in sentences:
        if s not in seen:
            seen.add(s)
            unique.append(s)

    steps = []

    for i in range(min(3, len(unique))):
        steps.append(f"Step {i+1}: {unique[i]}")

    return " ".join(steps) + "\nFinal Answer: Based on above reasoning."

In [4]:
def reward_function(text):
    score = 0

    if "step" in text.lower():
        score += 1
    if "final" in text.lower():
        score += 1
    if len(text) > 100:
        score += 0.5

    return score


def improve_reasoning(text):
    best = ""
    best_score = -1

    for _ in range(3):
        r = generate_rag_cot(text)
        s = reward_function(r)

        if s > best_score:
            best_score = s
            best = r

    return best

In [5]:
def final_system(text):
    prediction = predict_bail(text)

    reasoning_raw = improve_reasoning(text)
    reasoning = format_reasoning(reasoning_raw)

    important = get_important_sentences(text)

    return {
        "prediction": prediction,
        "reasoning": reasoning,
        "important_sentences": important
    }

In [19]:
trainer.save_model("finetuned_cot_model")
cot_tokenizer.save_pretrained("finetuned_cot_model")

('finetuned_cot_model/tokenizer_config.json',
 'finetuned_cot_model/special_tokens_map.json',
 'finetuned_cot_model/spiece.model',
 'finetuned_cot_model/added_tokens.json',
 'finetuned_cot_model/tokenizer.json')

In [20]:
cot_tokenizer = AutoTokenizer.from_pretrained("finetuned_cot_model")
cot_model = AutoModelForSeq2SeqLM.from_pretrained("finetuned_cot_model")

cot_model.to(device)
cot_model.eval()

T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=768, out_features=2048, bias=False)
              (wi_1): Linear(in_features=768, out_features=2048, bias=False)
              (wo):

In [21]:
def generate_rag_cot(text):
    retrieved = retrieve_chunks(text)
    context = " ".join(retrieved)

    important = get_important_sentences(text)
    facts = " ".join(important)

    prompt = f"""
You are a legal expert.

Facts:
{facts}

Context:
{context}

Generate reasoning:

Step 1:
Step 2:
Step 3:
Final Answer:
"""

    inputs = cot_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    ).to(device)

    outputs = cot_model.generate(
        **inputs,
        max_length=200,
        min_length=50,
        repetition_penalty=1.5,
        no_repeat_ngram_size=3
    )

    return cot_tokenizer.decode(outputs[0], skip_special_tokens=True)

In [22]:
sample = """
The accused is a 17-year-old juvenile involved in a robbery case.
He has no prior criminal record and has been in custody for 6 months.
The prosecution argued risk of reoffending, but no strong evidence was found.
"""

result = final_system(sample)

print("Prediction:", result["prediction"])
print("\nReasoning:\n", result["reasoning"])
print("\nImportant:\n", result["important_sentences"])

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Prediction: Granted

Reasoning:
 Step 1: Supreme Court ruled that the appellant should be granted bail under Sections 10, 13, 15, 16, 17, 18, 18, 18A, 18B, 19, 20, 23, and 38 of the Unlawful Activities (Prevention) Act, 1967 Step 2: Step 7 - Question: What was the main issue in the case of Jahir Hak vs The State of Rajasthan? Step 7
Final Answer: Based on above reasoning.

Important:
 ['\nThe accused is a 17-year-old juvenile involved in a robbery case', '\nHe has no prior criminal record and has been in custody for 6 months', '\nThe prosecution argued risk of reoffending, but no strong evidence was found']
